# RCC Subtype Classification - Mean Pooling MIL Baseline (5-class)
### BME 515 Final Project
5-class version using the same train/val/test split as CLAM experiments.
Serves as non-attention baseline for direct comparison against CLAM-SB and CLAM-MB.

Classes: Benign, Chromophobe, Clearcell, Oncocytoma, Papillary
Split files: split_train.csv, split_val.csv, split_test.csv (shared with CLAM)

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score
)

BASE_DIR    = Path('/Users/zhangruikun/Desktop/BME515')
FEATURE_DIR = BASE_DIR / 'features'
RESULT_DIR  = BASE_DIR / 'results'          # same folder as CLAM results
OUTPUT_DIR  = BASE_DIR / 'meanpool_5class_output'
OUTPUT_DIR.mkdir(exist_ok=True)

# split CSVs shared with CLAM — ensures identical train/val/test assignment
SPLIT_TRAIN = RESULT_DIR / 'split_train.csv'
SPLIT_VAL   = RESULT_DIR / 'split_val.csv'
SPLIT_TEST  = RESULT_DIR / 'split_test.csv'

CLASSES = ['Benign', 'Chromophobe', 'Clearcell', 'Oncocytoma', 'Papillary']

print(f'Feature dir exists: {FEATURE_DIR.exists()}')
print(f'Split train exists: {SPLIT_TRAIN.exists()}')

In [ ]:
# load split CSVs — train+val merged as training set, test for evaluation
train_df = pd.read_csv(SPLIT_TRAIN)
val_df   = pd.read_csv(SPLIT_VAL)
test_df  = pd.read_csv(SPLIT_TEST)

# merge train + val as training set (same approach as original 4-class baseline)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

print(f'Train+Val: {len(trainval_df)} slides | Test: {len(test_df)} slides')
print('\nTrain+Val class counts:')
print(trainval_df['Diagnosis'].value_counts())
print('\nTest class counts:')
print(test_df['Diagnosis'].value_counts())

In [ ]:
# build slide-level feature matrix via mean pooling
# for each slide: load patch features [N, 2048], average across patches -> [2048]
def load_mean_features(df, feature_dir):
    features, labels, slide_ids = [], [], []
    skipped = 0

    for _, row in df.iterrows():
        h5_path = feature_dir / f"{row['slide_id']}.h5"

        if not h5_path.exists():
            skipped += 1
            continue

        with h5py.File(h5_path, 'r') as f:
            feats = f['features'][:]  # [N, 2048]

        mean_feat = feats.mean(axis=0)  # [2048] — mean pooling

        features.append(mean_feat)
        labels.append(row['Diagnosis'])
        slide_ids.append(row['slide_id'])

    print(f'Loaded: {len(features)} slides | Skipped (missing h5): {skipped}')
    return np.stack(features), np.array(labels), slide_ids

X_train, y_train_raw, train_ids = load_mean_features(trainval_df, FEATURE_DIR)
X_test,  y_test_raw,  test_ids  = load_mean_features(test_df,     FEATURE_DIR)

print(f'\nX_train: {X_train.shape} | X_test: {X_test.shape}')

In [ ]:
# encode labels
le = LabelEncoder()
le.fit(CLASSES)  # fix label order to match CLAM
class_names = le.classes_

y_train = le.transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print(f'Classes: {class_names}')
print(f'Train class counts: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test class counts:  {dict(zip(*np.unique(y_test,  return_counts=True)))}')

In [ ]:
# normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print('Features normalized')

In [ ]:
# train SVM classifier
svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)
y_prob_svm = svm.predict_proba(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm  = f1_score(y_test, y_pred_svm, average='weighted')
auc_svm = roc_auc_score(y_test, y_prob_svm, multi_class='ovr', average='macro')

print(f'SVM — Accuracy: {acc_svm:.4f} | Weighted F1: {f1_svm:.4f} | Macro AUC: {auc_svm:.4f}')

In [ ]:
# train MLP classifier
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=300,
    early_stopping=True,
    random_state=42
)
mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
y_prob_mlp = mlp.predict_proba(X_test)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp  = f1_score(y_test, y_pred_mlp, average='weighted')
auc_mlp = roc_auc_score(y_test, y_prob_mlp, multi_class='ovr', average='macro')

print(f'MLP — Accuracy: {acc_mlp:.4f} | Weighted F1: {f1_mlp:.4f} | Macro AUC: {auc_mlp:.4f}')

In [ ]:
# confusion matrix — SVM
cm = confusion_matrix(y_test, y_pred_svm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Mean Pooling + SVM (5-class Baseline)')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix_meanpool5_svm.png'), dpi=150)
plt.show()
print('Saved confusion matrix')

In [ ]:
# per-class classification report
print('Classification Report — SVM:')
print(classification_report(y_test, y_pred_svm, target_names=class_names))

print('Classification Report — MLP:')
print(classification_report(y_test, y_pred_mlp, target_names=class_names))

In [ ]:
# summary table — SVM vs MLP vs CLAM-SB vs CLAM-MB
summary = pd.DataFrame({
    'Model':     ['Mean Pool + SVM', 'Mean Pool + MLP', 'CLAM-SB', 'CLAM-MB'],
    'Accuracy':  [round(acc_svm, 4), round(acc_mlp, 4), 0.767, 0.822],
    'W. F1':     [round(f1_svm,  4), round(f1_mlp,  4), 0.539, 0.599],
    'Macro AUC': [round(auc_svm, 4), round(auc_mlp, 4), 0.916, 0.939]
})
print('\n--- 5-class Comparison: Mean Pooling vs CLAM ---')
print(summary.to_string(index=False))
summary.to_csv(str(OUTPUT_DIR / 'comparison_5class.csv'), index=False)
print(f'\nSaved to {OUTPUT_DIR / "comparison_5class.csv"}')